In [3]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool,input_guardrail,GuardrailFunctionOutput
from sendgrid.helpers.mail import Mail, Email, To, Content
import os
import sendgrid
import asyncio 
from pydantic import BaseModel
from openai import AsyncOpenAI

In [4]:
load_dotenv(override=True)

True

In [5]:
openai_api_key = os.getenv("OPENAI_API_KEY")
google_api_key = os.getenv("GEMINI_API_KEY")
deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

In [6]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [7]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

In [8]:
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
deepseek_client = AsyncOpenAI(base_url=DEEPSEEK_BASE_URL, api_key=deepseek_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

In [9]:
from agents import OpenAIChatCompletionsModel


gemini_model = OpenAIChatCompletionsModel(model = "gemini-2.0-flash", openai_client = gemini_client)
deepseek_model = OpenAIChatCompletionsModel(model = "deepseek-chat", openai_client = deepseek_client)   
groq_model = OpenAIChatCompletionsModel(model = "llama3-70b-8192", openai_client = groq_client) 

In [45]:
prof_agent = Agent(name = "Professional Sales Agent", instructions=instructions1, model= 'gpt-4o-mini')
engg_agent = Agent(name = "Engaging Sales Agent", instructions=instructions2, model= deepseek_model)
busy_agent = Agent(name = "Busy Sales Agent", instructions=instructions1, model= gemini_model)


In [46]:
instructions_email_picker = "You pick the best cold sales email from the given options. \
Imagine you are a customer and pick the one you are most likely to respond to. \
Do not give an explanation; reply with the selected email only."
email_picker_agent = Agent(name = f"Best Sales Email Picker", instructions = instructions_email_picker, model = 'gpt-4o-mini')

In [47]:

@function_tool
def send_email_tool(body: str, subject :str):
    """ Send out an email with the given body to all sales prospects """
    
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("jigs.bagy@gmail.com")  # Change to your verified sender
    to_email = To("jigs.bagy@gmail.com")  # Change to your recipient
    content = Content("text/html", body)
    mail = Mail(from_email, to_email, subject, content).get()
    response = sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [48]:
toolDescription = "Write a cold sales email"
prof_email_tool = prof_agent.as_tool(tool_name = "Progessional_Email_Tool", tool_description = toolDescription)
engg_email_tool = engg_agent.as_tool(tool_name = "Engaging_Email_Tool", tool_description = toolDescription)
busy_email_tool = busy_agent.as_tool(tool_name = "Busy_Email_Tool", tool_description = toolDescription)
email_tools = [prof_email_tool, engg_email_tool, busy_email_tool]

In [49]:
subject_writer_instruction = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

subject_writer_agent = Agent(name = "Subject Writer Agent", instructions = subject_writer_instruction, model = "gpt-4o-mini")
subject_writer_tool = subject_writer_agent.as_tool(tool_name = "Subject_writer_tool", tool_description="Write a description for a cold sales email")

In [50]:

html_converter_instruction = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

html_converter_agent = Agent(name = "html conveter agent", instructions=html_converter_instruction, model = "gpt-4o-mini")
html_converter_tool = html_converter_agent.as_tool(tool_name = "html_converted_tool", tool_description="Convert simple text email body to html email body")

In [51]:
email_tools = [subject_writer_tool, html_converter_tool, send_email_tool ]

In [52]:
emailer_instruction = "You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."

email_agent = Agent(name = "Email Agent", instructions=  emailer_instruction, tools = email_tools, model = "gpt-4o-mini")

In [53]:
tools = [prof_email_tool, engg_email_tool, busy_email_tool]

In [54]:


sales_manager_instruction = "You are a sales manager working for ComplAI. You use the tools given to you to generate cold sales emails. \
You never generate sales emails yourself; you always use the tools. \
You try all 3 sales agent tools at least once before choosing the best one. \
You can use the tools multiple times if you're not satisfied with the results from the first try. \
You select the single best email using your own judgement of which email will be most effective. \
After picking the email, you handoff to the Email Manager agent to format and send the email."

sales_manager_agent = Agent(name = "Sales Manager Agent", 
    instructions= sales_manager_instruction, 
    tools = tools,
    handoffs= [email_agent],
    model = "gpt-4o-mini")

In [55]:
run_message = "Writer a cold email adddressed to Dear CEO from Jignesh"

with trace ("Cold Sales Email"):
    result = await Runner.run(sales_manager_agent,run_message)

run_message = "Write a cold sales email"
with trace("Write colde sales email"):
    results = await asyncio.gather(
        Runner.run(prof_agent, run_message),
        Runner.run(engg_agent, run_message),
        Runner.run(busy_agent, run_message)
    )

outputs = [result.final_output for result in results]

emails = "Colde Sales Emails : \n\n ".join(outputs)

best_email = await Runner.run(email_picker_agent, emails)

print (best_email.final_output)


sales_manager_instruction = 'instructions ="You are a sales manager working for ComplAI. You use the tools given to you to generate cold sales emails. \
You never generate sales emails yourself; you always use the tools. \
You try all 3 sales_agent tools once before choosing the best one. \
You pick the single best email and use the send_email tool to send the best email (and only the best email) to the user.'

sales_manager_agent = Agent(name = 'Sales Manager Agent', instructions = sales_manager_instruction, tools = email_tools, model = 'gpt-4o-mini')

sales_manager_agent_run_message = "Writer a cold email adddressed to Dear CEO"
with trace('Sales Manager Email'):
    email_ = await Runner.run(sales_manager_agent,sales_manager_agent_run_message)

In [28]:
class NameCheck(BaseModel):
    name: str
    is_valid: bool

name_check_agent = Agent(name = "Name Check Agent", 
instructions = "You are a name check agent. Check if user is inluding someone's name in whatever they want to do"
, model = "gpt-4o-mini"
, output_type = NameCheck
)


In [59]:
@input_guardrail
async def name_check_guardrail(ctx, agent, message):
    result = await Runner.run(agent, message, context = ctx.context)
    is_name_included = result.final_output.is_valid
    return GuardrailFunctionOutput(output_info = f"Name check result: {result.final_output}", tripwire_Triggered = not is_name_included)


In [60]:
carefull_sales_manager = Agent(name = "Care Full Sales Manager",
instructions= "sales_manager_instruction",
tools= tools,
handoffs=[email_agent],
model="gpt-4o-mini",
input_guardrails=[name_check_guardrail]
)

In [1]:
message = "send a cold sales email addressing to Dear CEO from John"

In [2]:
with trace("cold sales email"):
    result = await Runner.run(carefull_sales_manager, message)

NameError: name 'trace' is not defined